# Streamly — ¿quién va a cancelar el próximo mes?

Primer intento: un CSV, un notebook, un modelo.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv("../data/streamly_churn.csv")
df.shape

(90000, 19)

In [3]:
df.head()

   customer_id  age country  ... complaints failed_payments  churn
0            1   45      AR  ...          0               0      0
1            1   45      AR  ...          0               1      0
2            1   45      AR  ...          1               0      0
3            1   45      AR  ...          0               2      0
4            1   45      AR  ...          0               0      0

[5 rows x 19 columns]

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              90000 non-null  int64  
 1   age                      90000 non-null  int64  
 2   country                  90000 non-null  object 
 3   city                     90000 non-null  object 
 4   plan                     90000 non-null  object 
 5   monthly_price            90000 non-null  int64  
 6   payment_method           90000 non-null  object 
 7   mes                      90000 non-null  int64  
 8   subscription_months      90000 non-null  int64  
 9   sessions_last_30d        90000 non-null  int64  
 10  hours_watched            90000 non-null  float64
 11  unique_content_last_30d  90000 non-null  int64  
 12  completion_rate          90000 non-null  float64
 13  days_since_last_login    90000 non-null  int64  
 14  devices_used          

In [5]:
df.isna().sum().sum()

np.int64(0)

In [6]:
# cleaning
df = df.dropna(subset=["monthly_price", "days_since_last_login"])

In [7]:
# feature engineering
df["engagement_score"] = df["sessions_last_30d"] * df["completion_rate"]

In [8]:
df["churn"].value_counts(normalize=True)

churn
0    0.659656
1    0.340344
Name: proportion, dtype: float64

**Nota al margen:** cada cliente de Streamly aparece varias veces (una fila por mes). Con `random_state=42` y un split aleatorio, el mes 3 de un cliente puede quedar en train y su mes 4 en test — información del mismo cliente filtrándose entre los dos lados. Es un problema real (da para una charla completa con `GroupKFold`), pero hoy no es el que perseguimos. Seguimos.

In [9]:
# el modelo no entiende texto, así que las categóricas se codifican aquí mismo
X = pd.get_dummies(df.drop(columns=["churn", "customer_id"]))
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [11]:
print(roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))

0.760083640625085
